<a href="https://colab.research.google.com/github/arulperiyannagounder-collab/Training_Hands_on/blob/main/MCU_details_retrival_using_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-generativeai


In [ ]:
!pip install faiss-cpu
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
doc = [
  "Thanos, the 'Mad Titan', is a powerful cosmic warlord originating from the planet Titan. Created by Jim Starlin and Mike Friedrich, he debuted in Marvel Comics' Iron Man #55 in 1973. His origin and motivations differ slightly between the original comic books and the Marvel Cinematic Universe (MCU) films.Marvel Comics OriginIn the comics, Thanos is a member of the Eternals, a race of god-like beings created by cosmic entities known as the Celestials.The Deviant Gene: Thanos was born to A'Lars (Mentor) and Sui-San, but carried the 'Deviant gene,' which caused him to have purple, ridged skin and a larger, more menacing build than his peers. His mother was horrified by his appearance and even attempted to kill him.Obsession with Death: Ostracized by his people, he grew into an outcast and developed a nihilistic fascination with dying. This obsession manifested physically as a dark romance with Mistress Death, the living embodiment of death in the Marvel universe.The Infinity Stones: In the comics, Thanos's primary motivation for the Infinity Gauntlet is to gather the Infinity Stones to wipe out half of all life in the universe as an ultimate gesture of love and devotion to impress Mistress Death.",
  "Iron Man's origin centers on billionaire genius Tony Stark, who is critically injured by a weapon and captured by enemies. To survive, he builds a mechanized suit of armor and an arc reactor, transforming from a selfish weapons manufacturer into a heroic defender.",
  "Thor Odinson is the Asgardian God of Thunder and the heir to the throne of Asgard. Born over 1,500 years ago to Odin and Frigga, he began as an arrogant warrior whose brash actions reignited an ancient war, leading to his banishment to Earth to learn humility.",
  "Odin Borson is the former King of Asgard and the All-Father of the Nine Realms. The son of King Bor, Odin inherited the throne and the powerful, mystical Odinforce. He began his reign as a brutal conqueror before transitioning into a peaceful ruler dedicated to preserving universal stability.",
  "Howard Stark was a brilliant inventor, industrialist, and founder of Stark Industries. He was a pivotal figure in the Marvel Cinematic Universe who co-founded S.H.I.E.L.D., aided in the creation of the super-soldier serum, and built Captain America's vibranium shield. He and his wife Maria were assassinated by the Winter Soldier on December 16, 1991",
  "Odin's father in Marvel lore is Bor Burison (often referred to simply as King Bor). He is the first King of Asgard, son of Buri, and the grandfather of Thor and Hela.Key details about Bor in the Marvel Universe:Lineage & Reign: Bor led Asgard through a golden age of power, married the giantess Bestla, and fathered Odin as well as his brothers Cul, Vili, and Ve.Comic History (Earth-616): In the comics, Bor was among the foundational gods who created the universe. He was turned into 'living snow' by a sorcerer's trick during a battle, and Odin (who later established his own legacy) left him there to rule, allowing Bor to die. Bor was later resurrected to fight Thor, who had to defeat his own grandfather.MCU History (Earth-199999): In the Marvel Cinematic Universe (seen in the opening sequence of Thor: The Dark World), Bor led the Asgardians against Malekith and the Dark Elves, successfully securing the dangerous weapon known as the Aether"
]

model = SentenceTransformer('all-MiniLM-L6-V2')
doc_embedding = model.encode(doc).astype('float32')

dimension = doc_embedding.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embedding)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import google.generativeai as genai


genai.configure(api_key=GEMINI_API_KE)

query="who is Thor"
query_embed=model.encode([query]).astype("float32")
top_k=2
distance,indices=index.search(query_embed,top_k)
retrieved_chunk=[doc[i] for i in indices[0]]
print("Retrieved Chunks")
for j in retrieved_chunk:
  print('-',j)


Retrieved Chunks
- Thor Odinson is the Asgardian God of Thunder and the heir to the throne of Asgard. Born over 1,500 years ago to Odin and Frigga, he began as an arrogant warrior whose brash actions reignited an ancient war, leading to his banishment to Earth to learn humility.
- Odin's father in Marvel lore is Bor Burison (often referred to simply as King Bor). He is the first King of Asgard, son of Buri, and the grandfather of Thor and Hela.Key details about Bor in the Marvel Universe:Lineage & Reign: Bor led Asgard through a golden age of power, married the giantess Bestla, and fathered Odin as well as his brothers Cul, Vili, and Ve.Comic History (Earth-616): In the comics, Bor was among the foundational gods who created the universe. He was turned into 'living snow' by a sorcerer's trick during a battle, and Odin (who later established his own legacy) left him there to rule, allowing Bor to die. Bor was later resurrected to fight Thor, who had to defeat his own grandfather.MCU His

In [ ]:
context = "\n".join(retrieved_chunk)
prompt = f"""Answer the question based only on the context below.
Context: {context}
Question: {query}
Answer:"""

print("Final Answer:\n")


model_gemini = genai.GenerativeModel('gemini-flash-latest')
response = model_gemini.generate_content(prompt)

print(response.text)

Final Answer:

Based on the provided context, Thor (Thor Odinson) is the Asgardian God of Thunder and the heir to the throne of Asgard. He was born over 1,500 years ago to Odin and Frigga, and is the grandson of Bor. He began as an arrogant warrior whose brash actions reignited an ancient war, which led to him being banished to Earth to learn humility.
